In [7]:
import torchvision
from torchinfo import summary
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

def get_model():
    model = torchvision.models.video.r3d_18(pretrained=False)
    model.fc = torch.nn.Linear(in_features=512, out_features=2)
    return model

model = get_model().to(device)
summary(model)

cuda


/usr/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Layer (type:depth-idx)                   Param #
VideoResNet                              --
├─BasicStem: 1-1                         --
│    └─Conv3d: 2-1                       28,224
│    └─BatchNorm3d: 2-2                  128
│    └─ReLU: 2-3                         --
├─Sequential: 1-2                        --
│    └─BasicBlock: 2-4                   --
│    │    └─Sequential: 3-1              110,720
│    │    └─Sequential: 3-2              110,720
│    │    └─ReLU: 3-3                    --
│    └─BasicBlock: 2-5                   --
│    │    └─Sequential: 3-4              110,720
│    │    └─Sequential: 3-5              110,720
│    │    └─ReLU: 3-6                    --
├─Sequential: 1-3                        --
│    └─BasicBlock: 2-6                   --
│    │    └─Sequential: 3-7              221,440
│    │    └─Sequential: 3-8              442,624
│    │    └─ReLU: 3-9                    --
│    │    └─Sequential: 3-10             8,448
│    └─BasicBlock: 2-7           

In [1]:
# python
# monkeypatch configparser removed SafeConfigParser -> use ConfigParser
import configparser
if not hasattr(configparser, "SafeConfigParser"):
    configparser.SafeConfigParser = configparser.ConfigParser
import pylidc as pl

scans = pl.query(pl.Scan).all()

/home/stachu/projects/ood-detection/.venv/lib/python3.13/site-packages/pylidc/__init__.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as _pr


In [ ]:
from torch.utils.data import DataLoader, Dataset

class LIDCDataset(DataLoader):
    def __init__(self, scans: pl.Scan):
        self.scans = scans

    def __len__(self):
        return len(self.scans)
    
    def __getitem__(self, idx):
        scan = self.scans[idx]
        volume = scan.to_volume()
        label = 1 if len(scan.cluster_annotations) > 0 else 0
        volume = torch.tensor(volume, dtype=torch.float32).unsqueeze(0)  # add channel dimension
        return volume, label

dataset = LIDCDataset(scans)
train_dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

In [16]:
def train_model(model, dataloader, criterion, optimizer, device, num_epochs=3):
    model.to(device)
    model.train()

    for epoch in range(num_epochs):
        total_loss = 0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {total_loss / len(dataloader):.4f}")

In [17]:
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_model(model, train_dataloader, criterion, optimizer, device)

Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.
Loading dicom files ... This may take a moment.


RuntimeError: stack expects each tensor to be equal size, but got [1, 512, 512, 493] at entry 0 and [1, 512, 512, 280] at entry 1